# House Prices - Advanced Regression Techniques

## 1. Introduction

The goal of this project is to predict residential house sale prices using machine learning regression models.

This project explores two approaches to data preprocessing and model development:

- **Approach:** Manual data preprocessing and Ridge Regression without using Pipeline.

The performance of both approaches will be evaluated and compared using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).

Dataset: [Kaggle - House Prices: Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)

## 2. Import Libraries

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import Ridge

## 3. Load Dataset

In [4]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [5]:
train = pd.read_csv(
    "/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv"
)

sample_submission = pd.read_csv(
    "/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv"
)

In [6]:
print("Training data shape:", train.shape)
print("Test data shape:", test.shape)

display(train.head())

Training data shape: (1460, 81)
Test data shape: (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 4. Exploratory Data Analysis (EDA)

In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [8]:
missing_values = train.isnull().sum()

missing_values = missing_values[missing_values > 0].sort_values(
    ascending=False
)

display(missing_values)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

## 5. Data Preprocessing - Without Pipeline

In [10]:
X = train.drop(columns=["SalePrice", "Id"])
y = train["SalePrice"]

X_test = test.drop(columns=["Id"])

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Test features shape:", X_test.shape)

Features shape: (1460, 79)
Target shape: (1460,)
Test features shape: (1459, 79)


### Train-Validation Split

In [13]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

X_train: (1168, 79)
X_val: (292, 79)


### Identify Numerical and Categorical Features

In [14]:
numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns

print("Number of numerical features:", len(numerical_cols))
print("Number of categorical features:", len(categorical_cols))

Number of numerical features: 36
Number of categorical features: 43


In [15]:
X_train[numerical_cols] = X_train[numerical_cols].fillna(
    X_train[numerical_cols].median()
)

X_val[numerical_cols] = X_val[numerical_cols].fillna(
    X_train[numerical_cols].median()
)

X_test[numerical_cols] = X_test[numerical_cols].fillna(
    X_train[numerical_cols].median()
)

In [16]:
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence",
    "MasVnrType", "FireplaceQu",
    "GarageFinish", "GarageQual", "GarageType", "GarageCond",
    "BsmtFinType2", "BsmtExposure", "BsmtFinType1",
    "BsmtQual", "BsmtCond"
]

# Fill missing values with "None"
for col in none_cols:
    X_train[col] = X_train[col].fillna("None")
    X_val[col] = X_val[col].fillna("None")
    X_test[col] = X_test[col].fillna("None")

In [17]:
for col in categorical_cols:
    if col not in none_cols:
        mode_value = X_train[col].mode()[0]

        X_train[col] = X_train[col].fillna(mode_value)
        X_val[col] = X_val[col].fillna(mode_value)
        X_test[col] = X_test[col].fillna(mode_value)

In [18]:
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_val:", X_val.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

Missing values in X_train: 0
Missing values in X_val: 0
Missing values in X_test: 0


### One-Hot Encoding

Categorical features must be converted into numerical values.

We use One-Hot Encoding to create binary columns for each category:
- `1`: The observation belongs to the category.
- `0`: The observation does not belong to the category.

In [19]:
X_train = pd.get_dummies(
    X_train,
    columns=categorical_cols,
    dtype=int
)

X_val = pd.get_dummies(
    X_val,
    columns=categorical_cols,
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_cols,
    dtype=int
)

In [23]:
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (1168, 300)
Validation: (292, 300)
Test: (1459, 300)


In [24]:
print("Train columns:", len(X_train.columns))
print("Validation columns:", len(X_val.columns))
print("Test columns:", len(X_test.columns))

print("\nValidation extra columns:")
print(set(X_val.columns) - set(X_train.columns))

print("\nTest extra columns:")
print(set(X_test.columns) - set(X_train.columns))

Train columns: 300
Validation columns: 300
Test columns: 300

Validation extra columns:
set()

Test extra columns:
set()


In [26]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

## 6. Ridge Regression

Ridge Regression is a linear regression model that uses L2 regularization to reduce overfitting.

In [27]:
ridge_model = Ridge(alpha=10)

ridge_model.fit(X_train_scaled, y_train)

Ridge(alpha=10)

### Model Evaluation

We evaluate the model on the validation set using MAE and RMSE.

In [28]:
y_val_pred = ridge_model.predict(X_val_scaled)

mae = mean_absolute_error(y_val, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 18187.524220514875
RMSE: 28261.40265656437


### Hyperparameter Tuning

We test different alpha values to find the best regularization strength for Ridge Regression.

In [29]:
alphas = [0.1, 1, 10, 100, 1000]

for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)

    predictions = model.predict(X_val_scaled)

    mae = mean_absolute_error(y_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))

    print(f"alpha={alpha} | MAE={mae:.2f} | RMSE={rmse:.2f}")

alpha=0.1 | MAE=20054.01 | RMSE=47762.70
alpha=1 | MAE=18685.12 | RMSE=29732.49
alpha=10 | MAE=18187.52 | RMSE=28261.40
alpha=100 | MAE=18086.56 | RMSE=28541.83
alpha=1000 | MAE=19006.33 | RMSE=32449.50


### Selecting the Best Alpha

Based on the validation results, alpha=10 is selected because it achieves the lowest RMSE.

In [30]:
best_alpha = 10

ridge_model = Ridge(alpha=best_alpha)
ridge_model.fit(X_train_scaled, y_train)

Ridge(alpha=10)

### Test Predictions

The final Ridge Regression model is used to predict house prices in the test dataset.

In [31]:
test_predictions = ridge_model.predict(X_test_scaled)

print("Number of predictions:", len(test_predictions))
print("First 5 predictions:", test_predictions[:5])

Number of predictions: 1459
First 5 predictions: [119212.57057648 159640.30752043 187510.9077614  193651.55630839
 213409.92307976]


### Summary

The manual preprocessing approach with Ridge Regression achieved the following validation results:

- **Best Alpha:** 10
- **MAE:** 18,187.52
- **RMSE:** 28,261.40

The model successfully generated 1,459 predictions for the test dataset.